#### 15 PERCENT OVERSAMPLING USING ADASYN

In [1]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import ADASYN
from sklearn.model_selection import train_test_split
from collections import Counter
import os

# Load original reduced dataset
df = pd.read_csv("full_data_unhealthy_imputed_reduced_enhanced.csv")

# Define target and feature columns
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour_bin']

# Feature engineering
def create_features(X):
    X = X.copy()
    X['hour_sin'] = np.sin(2 * np.pi * X['hour_bin'] / 24)
    X['hour_cos'] = np.cos(2 * np.pi * X['hour_bin'] / 24)
    X['eat_rest_ratio'] = X['EAT'] / (X['REST'] + 1e-6)
    X['activity_rest_ratio'] = np.abs(X['ACTIVITY_LEVEL']) / (X['REST'] + 1e-6)
    return X.drop(columns=['hour_bin'])

X = create_features(df[features])
y = df[targets]

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Custom ADASYN resampling for 15% positive class ratio
def adasyn_resample_custom(X, y, target_ratio=0.15):
    X_res_list = []
    y_res_list = []

    for target in targets:
        print(f"\nResampling target: {target}")
        counts = Counter(y[target])
        majority_class = counts[0]
        desired_minority = int(majority_class * target_ratio / (1 - target_ratio))
        print(f"Original counts: {counts}, Desired minority: {desired_minority}")

        # Perform ADASYN oversampling
        sampler = ADASYN(sampling_strategy={1: desired_minority}, random_state=42, n_neighbors=5)
        X_res, y_res = sampler.fit_resample(X, y[target])

        X_res = pd.DataFrame(X_res, columns=X.columns)
        y_res = pd.Series(y_res, name=target)

        X_res_list.append(X_res)
        y_res_list.append(y_res)

    return X_res_list, y_res_list

# Run oversampling
X_resampled_list, y_resampled_list = adasyn_resample_custom(X_train, y_train, target_ratio=0.15)

# Save output CSVs
os.makedirs("adasyn_15pct", exist_ok=True)
for i, target in enumerate(targets):
    df_combined = pd.concat([X_resampled_list[i], y_resampled_list[i]], axis=1)
    df_combined.to_csv(f"adasyn_15pct/adasyn_15pct_{target}.csv", index=False)
    print(f"Saved: adasyn_15pct/adasyn_15pct_{target}.csv | Shape: {df_combined.shape}")



Resampling target: oestrus
Original counts: Counter({0: 253490, 1: 1042}), Desired minority: 44733

Resampling target: calving
Original counts: Counter({0: 253961, 1: 571}), Desired minority: 44816

Resampling target: lameness
Original counts: Counter({0: 254115, 1: 417}), Desired minority: 44843

Resampling target: mastitis
Original counts: Counter({0: 254406, 1: 126}), Desired minority: 44895
Saved: adasyn_15pct/adasyn_15pct_oestrus.csv | Shape: (297791, 9)
Saved: adasyn_15pct/adasyn_15pct_calving.csv | Shape: (298605, 9)
Saved: adasyn_15pct/adasyn_15pct_lameness.csv | Shape: (299004, 9)
Saved: adasyn_15pct/adasyn_15pct_mastitis.csv | Shape: (299352, 9)


#### RANDOM FOREST AND LIGHT GBM APPLIED

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import classification_report, make_scorer, f1_score
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
import joblib
import os

# 1. Configuration
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
data_dir = "adasyn_15pct"  # Folder containing the oversampled CSVs
model_dir = "models/15pct"
os.makedirs(model_dir, exist_ok=True)

# 2. Scoring and Cross-validation setup
scorer = make_scorer(lambda yt, yp: f1_score(yt, yp, average="weighted"))
cv = KFold(n_splits=3, shuffle=True, random_state=42)

# 3. Model training function
def tune_and_train(X_train, y_train, model_type):
    if model_type == "rf":
        base_model = RandomForestClassifier(random_state=42, n_jobs=-1)
        param_grid = {
            "n_estimators": [100, 150],
            "max_depth": [10, 20],
            "min_samples_split": [2, 5],
            "class_weight": ["balanced"]
        }
    else:  # LightGBM
        base_model = LGBMClassifier(random_state=42, n_jobs=-1)
        param_grid = {
            "n_estimators": [200, 300],
            "max_depth": [12, 16],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8],
            "colsample_bytree": [0.8],
            "class_weight": ["balanced"]
        }

    grid = GridSearchCV(
        base_model,
        param_grid,
        scoring=scorer,
        cv=cv,
        verbose=2,
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    print(f"Best params for {model_type.upper()}: {grid.best_params_}")
    return grid.best_estimator_

# 4. Train both RF and LGBM
for model_type in ["rf", "lgbm"]:
    print(f"\n=== Training {model_type.upper()} models with 15% ADASYN oversampling ===\n")

    for target in targets:
        print(f"--- {target} ---")
        # Load dataset
        df_res = pd.read_csv(f"{data_dir}/adasyn_15pct_{target}.csv")

        X = df_res.drop(columns=[target])
        y = df_res[target]

        # Train/test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        best_model = tune_and_train(X_train, y_train, model_type)

        y_pred = best_model.predict(X_test)
        print(f"\nClassification Report for {model_type.upper()} - {target}:\n")
        print(classification_report(y_test, y_pred))

        joblib.dump(best_model, f"{model_dir}/{model_type}_{target}.pkl")
        print(f"Saved model at {model_dir}/{model_type}_{target}.pkl")



=== Training RF models with 15% ADASYN oversampling ===

--- oestrus ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best params for RF: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 150}

Classification Report for RF - oestrus:

              precision    recall  f1-score   support

           0       0.97      0.96      0.97     50712
           1       0.78      0.85      0.82      8847

    accuracy                           0.94     59559
   macro avg       0.88      0.91      0.89     59559
weighted avg       0.95      0.94      0.94     59559

Saved model at models/15pct/rf_oestrus.pkl
--- calving ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best params for RF: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 100}

Classification Report for RF - calving:

              precision    recall  f1-score   support

           0       0.98      0.97      0.98     50852
     

#### MLP APPLIED

In [7]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader, TensorDataset

# Configuration
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
data_dir = "adasyn_15pct"
model_dir = "models/15pct_mlp"
os.makedirs(model_dir, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# MLP Model Definition
class MLPModel(nn.Module):
    def __init__(self, input_dim):
        super(MLPModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

# Training Function
def train_mlp(model, train_loader, val_loader, criterion, optimizer, epochs=20, patience=3):
    best_loss = float("inf")
    best_state = None
    counter = 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device).float().view(-1, 1)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device).float().view(-1, 1)
                preds = model(xb)
                val_loss += criterion(preds, yb).item()

        avg_val_loss = val_loss / len(val_loader)

        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            best_state = model.state_dict()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    if best_state:
        model.load_state_dict(best_state)

    return model

# Training Loop for all targets
for target in targets:
    print(f"\n=== Training MLP for target: {target} (15% ADASYN) ===")

    df = pd.read_csv(f"{data_dir}/adasyn_15pct_{target}.csv")
    X = df.drop(columns=[target]).values
    y = df[target].values

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    # Prepare PyTorch DataLoaders
    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                             torch.tensor(y_train, dtype=torch.float32))
    val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                           torch.tensor(y_val, dtype=torch.float32))
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=128)

    model = MLPModel(input_dim=X.shape[1]).to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model = train_mlp(model, train_loader, val_loader, criterion, optimizer)

    # Evaluation
    model.eval()
    with torch.no_grad():
        y_pred = []
        for xb, _ in val_loader:
            xb = xb.to(device)
            preds = model(xb).cpu().numpy()
            y_pred.extend((preds > 0.5).astype(int).flatten())

    print(classification_report(y_val, y_pred, digits=4))

    # Save model
    torch.save(model.state_dict(), f"{model_dir}/mlp_{target}_15pct.pth")
    print(f"Saved model to {model_dir}/mlp_{target}_15pct.pth")



=== Training MLP for target: oestrus (15% ADASYN) ===
Early stopping at epoch 6
              precision    recall  f1-score   support

           0     0.8689    0.9870    0.9242     50712
           1     0.6627    0.1462    0.2395      8847

    accuracy                         0.8621     59559
   macro avg     0.7658    0.5666    0.5818     59559
weighted avg     0.8383    0.8621    0.8225     59559

Saved model to models/15pct_mlp/mlp_oestrus_15pct.pth

=== Training MLP for target: calving (15% ADASYN) ===
Early stopping at epoch 9
              precision    recall  f1-score   support

           0     0.9240    0.9756    0.9491     50852
           1     0.7941    0.5397    0.6427      8869

    accuracy                         0.9109     59721
   macro avg     0.8591    0.7577    0.7959     59721
weighted avg     0.9047    0.9109    0.9036     59721

Saved model to models/15pct_mlp/mlp_calving_15pct.pth

=== Training MLP for target: lameness (15% ADASYN) ===
Early stopping at ep

C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0     0.8484    1.0000    0.9180     50737
           1     0.0000    0.0000    0.0000      9064

    accuracy                         0.8484     59801
   macro avg     0.4242    0.5000    0.4590     59801
weighted avg     0.7198    0.8484    0.7789     59801

Saved model to models/15pct_mlp/mlp_lameness_15pct.pth

=== Training MLP for target: mastitis (15% ADASYN) ===
Early stopping at epoch 3
              precision    recall  f1-score   support

           0     0.8488    1.0000    0.9182     50816
           1     0.0000    0.0000    0.0000      9055

    accuracy                         0.8488     59871
   macro avg     0.4244    0.5000    0.4591     59871
weighted avg     0.7204    0.8488    0.7793     59871

Saved model to models/15pct_mlp/mlp_mastitis_15pct.pth


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### TABNET APPLIED

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
import torch
import os

# ----------------------
# Setup
# ----------------------
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
data_dir = "adasyn_15pct"
model_dir = "models/15pct_tabnet"
os.makedirs(model_dir, exist_ok=True)

# ----------------------
# Training Loop
# ----------------------
for target in targets:
    print(f"\n=== Training TabNet for target: {target} (15% ADASYN) ===")
    
    # Load data
    df = pd.read_csv(f"{data_dir}/adasyn_15pct_{target}.csv")
    X = df.drop(columns=[target])
    y = df[target]

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Encode target
    y_train_enc = LabelEncoder().fit_transform(y_train)
    y_test_enc = LabelEncoder().fit_transform(y_test)

    # Define TabNet model
    clf = TabNetClassifier(
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=1e-2),
        scheduler_params={"step_size":10, "gamma":0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        verbose=0,
        seed=42
    )

    # Train model
    clf.fit(
        X_train=X_train_scaled, y_train=y_train_enc,
        eval_set=[(X_test_scaled, y_test_enc)],
        eval_name=["val"],
        eval_metric=["accuracy"],
        max_epochs=100,
        patience=10,
        batch_size=1024,
        virtual_batch_size=128
    )

    # Predict and evaluate
    preds = clf.predict(X_test_scaled)
    report = classification_report(y_test_enc, preds, target_names=['0', '1'])
    print(report)

    # Save model
    clf.save_model(f"{model_dir}/tabnet_{target}_15pct")



=== Training TabNet for target: oestrus (15% ADASYN) ===

Early stopping occurred at epoch 13 with best_epoch = 3 and best_val_accuracy = 0.91523


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


              precision    recall  f1-score   support

           0       0.91      0.99      0.95     50712
           1       0.93      0.46      0.62      8847

    accuracy                           0.92     59559
   macro avg       0.92      0.73      0.79     59559
weighted avg       0.92      0.92      0.90     59559

Successfully saved model at models/15pct_tabnet/tabnet_oestrus_15pct.zip

=== Training TabNet for target: calving (15% ADASYN) ===

Early stopping occurred at epoch 20 with best_epoch = 10 and best_val_accuracy = 0.94717


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


              precision    recall  f1-score   support

           0       0.95      0.99      0.97     50852
           1       0.92      0.71      0.80      8869

    accuracy                           0.95     59721
   macro avg       0.94      0.85      0.88     59721
weighted avg       0.95      0.95      0.94     59721

Successfully saved model at models/15pct_tabnet/tabnet_calving_15pct.zip

=== Training TabNet for target: lameness (15% ADASYN) ===

Early stopping occurred at epoch 15 with best_epoch = 5 and best_val_accuracy = 0.92045


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


              precision    recall  f1-score   support

           0       0.92      1.00      0.96     50737
           1       0.97      0.49      0.65      9064

    accuracy                           0.92     59801
   macro avg       0.95      0.74      0.80     59801
weighted avg       0.92      0.92      0.91     59801

Successfully saved model at models/15pct_tabnet/tabnet_lameness_15pct.zip

=== Training TabNet for target: mastitis (15% ADASYN) ===

Early stopping occurred at epoch 22 with best_epoch = 12 and best_val_accuracy = 0.91891


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


              precision    recall  f1-score   support

           0       0.91      1.00      0.95     50816
           1       0.98      0.47      0.64      9055

    accuracy                           0.92     59871
   macro avg       0.95      0.74      0.80     59871
weighted avg       0.92      0.92      0.91     59871

Successfully saved model at models/15pct_tabnet/tabnet_mastitis_15pct.zip


#### LSTM APPLIED

In [13]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

# Define LSTM model
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # last time step
        out = self.fc(out)
        return self.sigmoid(out)

# Training function
def train_lstm_model(X, y, target_name, model_save_path):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = np.expand_dims(X_scaled, axis=1)  # Reshape for LSTM

    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

    train_data = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train.values, dtype=torch.float32))
    test_data = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test.values, dtype=torch.float32))

    train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=64)

    model = LSTMClassifier(input_dim=X_train.shape[2], hidden_dim=64, output_dim=1)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(10):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            preds = model(xb).squeeze()
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

        # Evaluation
        model.eval()
        preds_all, labels_all = [], []
        with torch.no_grad():
            for xb, yb in test_loader:
                out = model(xb).squeeze()
                preds_all.extend((out > 0.5).int().cpu().numpy())
                labels_all.extend(yb.int().cpu().numpy())

        print(f"\nEpoch {epoch+1} - {target_name}:\n")
        print(classification_report(labels_all, preds_all))

    # Save model
    model_path = os.path.join(model_save_path, f"lstm_{target_name}_15pct.pth")
    torch.save(model.state_dict(), model_path)
    print(f"✅ Model saved to: {model_path}\n")
    return model_path

# Main training loop
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
os.makedirs("models/15pct_lstm", exist_ok=True)

for target in targets:
    df = pd.read_csv(f"adasyn_15pct/adasyn_15pct_{target}.csv")
    X = df.drop(columns=[target])
    y = df[target]
    train_lstm_model(X, y, target, model_save_path="models/15pct_lstm")



Epoch 1 - oestrus:

              precision    recall  f1-score   support

           0       0.91      0.99      0.95     50712
           1       0.87      0.41      0.55      8847

    accuracy                           0.90     59559
   macro avg       0.89      0.70      0.75     59559
weighted avg       0.90      0.90      0.89     59559


Epoch 2 - oestrus:

              precision    recall  f1-score   support

           0       0.91      0.99      0.95     50712
           1       0.94      0.45      0.61      8847

    accuracy                           0.91     59559
   macro avg       0.93      0.72      0.78     59559
weighted avg       0.92      0.91      0.90     59559


Epoch 3 - oestrus:

              precision    recall  f1-score   support

           0       0.91      1.00      0.95     50712
           1       0.97      0.43      0.59      8847

    accuracy                           0.91     59559
   macro avg       0.94      0.71      0.77     59559
weighted av

#### TCN APPLIED

In [1]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.optim import Adam
from torch.nn.utils import weight_norm
import random

# Fix seed for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Define your dataset
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# TCN block
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout):
        super().__init__()
        self.conv1 = weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)

        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
        self.init_weights()

    def init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight)

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCN(nn.Module):
    def __init__(self, input_size, output_size, num_channels, kernel_size=3, dropout=0.2):
        super().__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = input_size if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1, dilation=dilation_size,
                                     padding=(kernel_size-1)*dilation_size, dropout=dropout)]
        self.network = nn.Sequential(*layers)
        self.linear = nn.Linear(num_channels[-1], output_size)

    def forward(self, x):
        y1 = self.network(x)
        y1 = y1[:, :, -1]  # Last time step
        out = self.linear(y1)
        return out

# Load and prepare data
def train_tcn_model(target):
    data_path = f'adasyn_15pct/adasyn_15pct_{target}.csv'
    df = pd.read_csv(data_path)
    features = df.drop(columns=[target])
    labels = df[target]

    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    
    # Reshape to [samples, channels, sequence_length]
    X = features_scaled.reshape(features_scaled.shape[0], features_scaled.shape[1], 1)
    X = np.transpose(X, (0, 2, 1))  # [batch, channels, time]

    X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42, stratify=labels)

    train_loader = DataLoader(TimeSeriesDataset(X_train, y_train), batch_size=256, shuffle=True)
    test_loader = DataLoader(TimeSeriesDataset(X_test, y_test), batch_size=256, shuffle=False)

    model = TCN(input_size=1, output_size=2, num_channels=[16, 32, 64], kernel_size=3, dropout=0.3)
    model = model.cuda() if torch.cuda.is_available() else model
    criterion = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=0.001)

    best_accuracy = 0
    for epoch in range(1, 11):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch, y_batch
            if torch.cuda.is_available():
                X_batch, y_batch = X_batch.cuda(), y_batch.cuda()
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        model.eval()
        y_true, y_pred = [], []
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                if torch.cuda.is_available():
                    X_batch = X_batch.cuda()
                outputs = model(X_batch)
                _, predicted = torch.max(outputs, 1)
                y_true.extend(y_batch.numpy())
                y_pred.extend(predicted.cpu().numpy())

        report = classification_report(y_true, y_pred, digits=4)
        print(f"\nEpoch {epoch} - {target}:\n")
        print(report)

        acc = (np.array(y_pred) == np.array(y_true)).mean()
        if acc > best_accuracy:
            best_accuracy = acc
            save_path = f'models/15pct_tcn/tcn_{target}_15pct.pth'
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            torch.save(model.state_dict(), save_path)
            print(f"✅ Model saved to: {save_path}")

# Train for all 4 targets
for target_name in ['oestrus', 'calving', 'lameness', 'mastitis']:
    train_tcn_model(target_name)


C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



Epoch 1 - oestrus:

              precision    recall  f1-score   support

           0     0.9125    0.9833    0.9466     50699
           1     0.8278    0.4605    0.5918      8860

    accuracy                         0.9055     59559
   macro avg     0.8701    0.7219    0.7692     59559
weighted avg     0.8999    0.9055    0.8938     59559

✅ Model saved to: models/15pct_tcn/tcn_oestrus_15pct.pth

Epoch 2 - oestrus:

              precision    recall  f1-score   support

           0     0.9092    0.9948    0.9501     50699
           1     0.9354    0.4317    0.5908      8860

    accuracy                         0.9110     59559
   macro avg     0.9223    0.7133    0.7704     59559
weighted avg     0.9131    0.9110    0.8966     59559

✅ Model saved to: models/15pct_tcn/tcn_oestrus_15pct.pth

Epoch 3 - oestrus:

              precision    recall  f1-score   support

           0     0.9150    0.9858    0.9491     50699
           1     0.8539    0.4757    0.6110      8860

    a

C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



Epoch 1 - calving:

              precision    recall  f1-score   support

           0     0.9405    0.9882    0.9638     50792
           1     0.9057    0.6443    0.7530      8929

    accuracy                         0.9368     59721
   macro avg     0.9231    0.8163    0.8584     59721
weighted avg     0.9353    0.9368    0.9322     59721

✅ Model saved to: models/15pct_tcn/tcn_calving_15pct.pth

Epoch 2 - calving:

              precision    recall  f1-score   support

           0     0.9473    0.9882    0.9673     50792
           1     0.9113    0.6871    0.7835      8929

    accuracy                         0.9432     59721
   macro avg     0.9293    0.8377    0.8754     59721
weighted avg     0.9419    0.9432    0.9398     59721

✅ Model saved to: models/15pct_tcn/tcn_calving_15pct.pth

Epoch 3 - calving:

              precision    recall  f1-score   support

           0     0.9446    0.9873    0.9655     50792
           1     0.9030    0.6707    0.7697      8929

    a

C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



Epoch 1 - lameness:

              precision    recall  f1-score   support

           0     0.9123    0.9761    0.9431     50823
           1     0.7758    0.4689    0.5845      8978

    accuracy                         0.8999     59801
   macro avg     0.8440    0.7225    0.7638     59801
weighted avg     0.8918    0.8999    0.8893     59801

✅ Model saved to: models/15pct_tcn/tcn_lameness_15pct.pth

Epoch 2 - lameness:

              precision    recall  f1-score   support

           0     0.9160    0.9807    0.9473     50823
           1     0.8179    0.4912    0.6138      8978

    accuracy                         0.9072     59801
   macro avg     0.8670    0.7359    0.7805     59801
weighted avg     0.9013    0.9072    0.8972     59801

✅ Model saved to: models/15pct_tcn/tcn_lameness_15pct.pth

Epoch 3 - lameness:

              precision    recall  f1-score   support

           0     0.9199    0.9825    0.9501     50823
           1     0.8386    0.5155    0.6385      8978



C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



Epoch 1 - mastitis:

              precision    recall  f1-score   support

           0     0.9070    0.9963    0.9495     50882
           1     0.9523    0.4217    0.5846      8989

    accuracy                         0.9100     59871
   macro avg     0.9296    0.7090    0.7671     59871
weighted avg     0.9138    0.9100    0.8947     59871

✅ Model saved to: models/15pct_tcn/tcn_mastitis_15pct.pth

Epoch 2 - mastitis:

              precision    recall  f1-score   support

           0     0.9095    0.9976    0.9515     50882
           1     0.9702    0.4382    0.6037      8989

    accuracy                         0.9136     59871
   macro avg     0.9399    0.7179    0.7776     59871
weighted avg     0.9186    0.9136    0.8993     59871

✅ Model saved to: models/15pct_tcn/tcn_mastitis_15pct.pth

Epoch 3 - mastitis:

              precision    recall  f1-score   support

           0     0.9125    0.9985    0.9536     50882
           1     0.9824    0.4582    0.6249      8989



#### HYBRID ENSEMBLE METHOD

In [39]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import joblib
from pytorch_tabnet.tab_model import TabNetClassifier
import pickle

# ----------------------
# Configuration
# ----------------------
SEQ_LEN = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGETS = ["oestrus", "calving", "lameness", "mastitis"]
FEATURES_8 = ['IN_ALLEYS','REST','EAT','ACTIVITY_LEVEL','hour_sin','hour_cos','eat_rest_ratio','activity_rest_ratio']

model_dirs = {
    "rf": "models/15pct",
    "lgbm": "models/15pct",
    "lstm": "models/15pct_lstm",
    "tcn": "models/15pct_tcn",
    "tabnet": "models/15pct_tabnet",
    "ensemble": "models/15pct_ensemble"
}

# ----------------------
# Helper Classes
# ----------------------
class SeqDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx]

# LSTM model class
class LSTMModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=64, batch_first=True)
        self.fc = nn.Linear(64, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

# TCN architecture from training
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout):
        super().__init__()
        self.conv1 = nn.utils.weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size,
                                                    stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.utils.weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size,
                                                    stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCNModel(nn.Module):
    def __init__(self, input_size=1, output_size=2, num_channels=[16, 32, 64], kernel_size=3, dropout=0.3):
        super().__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation_size = 2 ** i
            in_channels = input_size if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers.append(TemporalBlock(in_channels, out_channels, kernel_size, stride=1,
                                        dilation=dilation_size, padding=(kernel_size-1)*dilation_size,
                                        dropout=dropout))
        self.network = nn.Sequential(*layers)
        self.linear = nn.Linear(num_channels[-1], output_size)

    def forward(self, x):
        x = self.network(x)
        x = x[:, :, -1]  # Last time step
        return self.linear(x)

# ----------------------
# Inference Loop
# ----------------------
for target in TARGETS:
    print(f"\n--- Hybrid Ensemble (15% ADASYN) for: {target} ---")

    # Load Data
    df = pd.read_csv(f"adasyn_15pct/adasyn_15pct_{target}.csv")
    X = df[FEATURES_8]
    y = df[target]

    # Label encode target
    le = LabelEncoder()
    y_enc = le.fit_transform(y)

    # Split
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, stratify=y_enc, random_state=42)

    # Tabular features
    rf_path = os.path.join(model_dirs['rf'], f"rf_{target}.pkl")
    lgbm_path = os.path.join(model_dirs['lgbm'], f"lgbm_{target}.pkl")
    tabnet_path = os.path.join(model_dirs['tabnet'], f"tabnet_{target}_15pct.zip")

    if not os.path.exists(rf_path):
        print(f"❌ RF model not found: {rf_path}")
        continue

    if not os.path.exists(lgbm_path):
        print(f"❌ LGBM model not found: {lgbm_path}")
        continue

    rf = joblib.load(rf_path)
    lgbm = joblib.load(lgbm_path)
    tabnet = TabNetClassifier()
    tabnet.load_model(tabnet_path)

    rf_probs = rf.predict_proba(X_test)[:, 1]
    lgbm_probs = lgbm.predict_proba(X_test)[:, 1]
    tabnet_probs = tabnet.predict_proba(X_test.values)[:, 1]

    # LSTM sequence
    X_seq = X.values.reshape(-1, 1, X.shape[1])
    X_seq_tensor = torch.tensor(X_seq, dtype=torch.float32).to(DEVICE)

    lstm_path = os.path.join(model_dirs['lstm'], f"lstm_{target}_15pct.pth")
    lstm_model = LSTMModel(input_size=X.shape[1])
    lstm_model.load_state_dict(torch.load(lstm_path, map_location=DEVICE))
    lstm_model.to(DEVICE).eval()
    with torch.no_grad():
        lstm_logits = lstm_model(X_seq_tensor)
        lstm_probs = torch.sigmoid(lstm_logits).cpu().numpy().squeeze()

    # Load TCN
    tcn = TCNModel()
    tcn_path = os.path.join(model_dirs['tcn'], f"tcn_{target}_15pct.pth")
    tcn.load_state_dict(torch.load(tcn_path, map_location=DEVICE))
    tcn.to(DEVICE).eval()
    with torch.no_grad():
        tcn_probs = tcn(X_seq_tensor).softmax(dim=1).cpu().numpy()[:, 1]

    # Combine predictions
    min_len = min(len(rf_probs), len(lgbm_probs), len(tabnet_probs), len(lstm_probs), len(tcn_probs))
    final_prob = (
        rf_probs[:min_len] +
        lgbm_probs[:min_len] +
        tabnet_probs[:min_len] +
        lstm_probs[:min_len] +
        tcn_probs[:min_len]
    ) / 5

    final_preds = (final_prob > 0.5).astype(int)
    y_true = y_test[:min_len]

    print(classification_report(y_true, final_preds, digits=4))

    # Save the ensemble model output
    os.makedirs(model_dirs['ensemble'], exist_ok=True)
    output_path = os.path.join(model_dirs['ensemble'], f"ensemble_preds_{target}.pkl")
    with open(output_path, "wb") as f:
        pickle.dump({
            "y_true": y_true,
            "y_pred": final_preds,
            "y_prob": final_prob
        }, f)
    print(f"✅ Saved ensemble predictions for {target} to: {output_path}")



--- Hybrid Ensemble (15% ADASYN) for: oestrus ---


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


              precision    recall  f1-score   support

           0     0.9542    0.9616    0.9579     50699
           1     0.7702    0.7357    0.7525      8860

    accuracy                         0.9280     59559
   macro avg     0.8622    0.8487    0.8552     59559
weighted avg     0.9268    0.9280    0.9273     59559

✅ Saved ensemble predictions for oestrus to: models/15pct_ensemble\ensemble_preds_oestrus.pkl

--- Hybrid Ensemble (15% ADASYN) for: calving ---


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


              precision    recall  f1-score   support

           0     0.9918    0.7501    0.8542     50792
           1     0.4043    0.9648    0.5698      8929

    accuracy                         0.7822     59721
   macro avg     0.6981    0.8575    0.7120     59721
weighted avg     0.9040    0.7822    0.8117     59721

✅ Saved ensemble predictions for calving to: models/15pct_ensemble\ensemble_preds_calving.pkl

--- Hybrid Ensemble (15% ADASYN) for: lameness ---


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


              precision    recall  f1-score   support

           0     0.8914    0.9910    0.9386     50823
           1     0.8616    0.3168    0.4632      8978

    accuracy                         0.8898     59801
   macro avg     0.8765    0.6539    0.7009     59801
weighted avg     0.8869    0.8898    0.8672     59801

✅ Saved ensemble predictions for lameness to: models/15pct_ensemble\ensemble_preds_lameness.pkl

--- Hybrid Ensemble (15% ADASYN) for: mastitis ---


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


              precision    recall  f1-score   support

           0     0.9613    0.9906    0.9757     50882
           1     0.9356    0.7743    0.8473      8989

    accuracy                         0.9581     59871
   macro avg     0.9485    0.8824    0.9115     59871
weighted avg     0.9574    0.9581    0.9564     59871

✅ Saved ensemble predictions for mastitis to: models/15pct_ensemble\ensemble_preds_mastitis.pkl
